# 025번 고령자 근현대 경험 기반 스토리 구술 데이터 탐색

**목적**: `extract-memory` 카테고리 체계 재정의 근거 마련

**Drive 경로**: `내드라이브/Dadam_dataSet/025.고령자 근현대 경험 기반 스토리 구술 데이터/data/data/Validation/라벨링데이터/`

**데이터 구조**:
- zip 50개 (주제별): `VL_XX.대분류_NNN.주제명.zip`
- 대분류 5개: 01·02·03·04·05
- 각 zip 내 JSON 200개 (사람 ID 기반)

**활용처**:
- TASK-03: `extract-memory` 카테고리 7개 재정의 (대분류 5개 → 7카테고리 압축 매핑)
- TASK-05: `generate-book` 챕터 구성 기준 (감정→장소→관계 흐름)

## 처리 흐름
```
Cell 1 → Drive 마운트
Cell 2 → 경로 설정 및 zip 파일 목록 확인
Cell 3 → JSON 구조 파악 (zip 1개 샘플)
Cell 4 → 전체 zip 대분류별 샘플 수집 (각 zip 1개 JSON)
Cell 5 → 발화 패턴 분석 (주제별 키워드·길이 분포)
Cell 6 → 카테고리 매핑 기준 도출 및 예시 선별
Cell 7 → extract-memory 카테고리 few-shot JSON 저장
Cell 8 → 로컬 다운로드
```

## Cell 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2. 경로 설정 및 zip 파일 목록 확인

zip 50개 전체 목록을 대분류별로 정리해 다음 셀에서 처리할 준비.

In [ ]:
import os
import zipfile
from collections import defaultdict

# Drive 경로
LABEL_DIR = '/content/drive/MyDrive/Dadam_dataSet/025.고령자 근현대 경험 기반 스토리 구술 데이터/data/data/Validation/라벨링데이터'

print('경로 존재 여부:', os.path.exists(LABEL_DIR))

# 경로 없으면 자동 탐색
if not os.path.exists(LABEL_DIR):
    MYDRIVE = '/content/drive/MyDrive'
    print('[자동 탐색] 025 폴더 검색 중...')
    for dirpath, dirnames, filenames in os.walk(MYDRIVE):
        depth = dirpath.replace(MYDRIVE, '').count(os.sep)
        if depth > 7:
            del dirnames[:]
            continue
        # 라벨링데이터 폴더 안에 VL_0*.zip 있는지 확인
        zips = [f for f in filenames if f.startswith('VL_') and f.endswith('.zip')]
        if zips:
            LABEL_DIR = dirpath
            print(f'발견: {LABEL_DIR}')
            break

# zip 파일 목록 수집
all_zips = sorted([f for f in os.listdir(LABEL_DIR) if f.endswith('.zip')])
print(f'\nzip 파일 총 {len(all_zips)}개')

# 대분류별 분류
# 파일명 패턴: VL_05.관계 및 사건_050.여행.zip
category_map = defaultdict(list)
for fname in all_zips:
    # VL_XX에서 XX 추출
    prefix = fname.split('.')[0]  # 'VL_05'
    cat_num = prefix.split('_')[1]  # '05'
    category_map[cat_num].append(fname)

print('\n=== 대분류별 zip 목록 ===')
for cat_num in sorted(category_map.keys()):
    zips = category_map[cat_num]
    # 대분류명 추출 (예: 'VL_05.관계 및 사건_050.여행.zip' → '관계 및 사건')
    cat_name = all_zips[0]  # fallback
    for z in zips:
        parts = z.split('.')
        if len(parts) >= 2:
            cat_name = parts[1].split('_')[0]  # '관계 및 사건'
            break
    # 대분류명을 첫 zip에서 추출
    first_zip = zips[0]
    # VL_05.관계 및 사건_050.여행.zip → split('.') = ['VL_05', '관계 및 사건_050', '여행', 'zip']
    try:
        cat_name = first_zip.split('.')[1].rsplit('_', 1)[0]
    except Exception:
        cat_name = '?'
    print(f'  [{cat_num}] {cat_name} — {len(zips)}개 주제')
    for z in zips:
        topic = z.split('.')[-2] if '.' in z else z
        print(f'    - {z}')

## Cell 3. JSON 구조 파악

zip 1개를 열어 JSON 내부 구조를 확인.

예상 구조:
```json
{
  "id": "...",
  "speaker": { "age": ..., "gender": ... },
  "topic": "여행",
  "utterances": [
    { "speaker": "interviewer", "text": "..." },
    { "speaker": "subject",     "text": "..." },
    ...
  ]
}
```

→ 실제 키 확인 후 파싱 코드 결정

In [ ]:
import json

# VL_05.관계 및 사건_050.여행.zip 열어서 첫 번째 JSON 구조 확인
# (스크린샷 기준 목록 첫 번째 zip)
sample_zip_name = all_zips[0]
sample_zip_path = os.path.join(LABEL_DIR, sample_zip_name)
print(f'샘플 zip: {sample_zip_name}')

with zipfile.ZipFile(sample_zip_path, 'r') as z:
    json_files_in_zip = [f for f in z.namelist() if f.endswith('.json')]
    print(f'zip 내 JSON 수: {len(json_files_in_zip)}개')
    print('파일명 샘플:', json_files_in_zip[:3])

    with z.open(json_files_in_zip[0]) as jf:
        raw = json.load(jf)

# 최상위 타입 확인
print('\n최상위 타입:', type(raw).__name__)

if isinstance(raw, dict):
    print('최상위 키:', list(raw.keys()))
    for k, v in raw.items():
        if isinstance(v, list):
            print(f'  [{k}] 리스트 {len(v)}개')
            if v and isinstance(v[0], dict):
                print(f'    첫 항목 키:', list(v[0].keys()))
                print(f'    첫 항목 내용:', v[0])
        elif isinstance(v, dict):
            print(f'  [{k}] dict:', v)
        else:
            print(f'  [{k}]:', v)
elif isinstance(raw, list):
    print(f'리스트 {len(raw)}개')
    if raw and isinstance(raw[0], dict):
        print('첫 항목 키:', list(raw[0].keys()))
        for k, v in raw[0].items():
            print(f'  [{k}]:', v if not isinstance(v, (list, dict)) else type(v).__name__)

## Cell 4. 전체 zip 대분류별 샘플 수집

50개 zip을 순회하면서 각 zip의 JSON 1~3개를 열어:
- 구술자(노인) 발화 텍스트
- 대분류 번호·이름
- 주제명

을 수집. 데이터가 많으므로 zip당 JSON 2개씩만 샘플링.

In [ ]:
import pandas as pd
import re

# 025번 실제 구조:
# {'qa': [{'question': '...', 'answer': '...'}], 'teller': [...], 'keyword': '기쁘다', ...}
# 구술자 발화 = qa[0]['answer']
# 라벨 = label_1[0] (사건구체성·시간적구체성·공간적구체성·주관적경험·자서전적기억)
# 화자 정보 = teller[0] (나이, 성별 등)

def extract_subject_utterances(raw):
    """025번 JSON에서 구술자 발화(qa[0].answer) 추출"""
    if not isinstance(raw, dict):
        return []
    qa_list = raw.get('qa', [])
    if not qa_list:
        return []
    answer = qa_list[0].get('answer', '').strip()
    return [answer] if answer else []

def extract_row(raw, cat_num, cat_name, topic, zip_fname, jf_name):
    """JSON 1개에서 분석에 필요한 필드 추출"""
    if not isinstance(raw, dict):
        return None

    qa_list = raw.get('qa', [])
    if not qa_list:
        return None
    answer = qa_list[0].get('answer', '').strip()
    if not answer:
        return None

    teller = raw.get('teller', [{}])[0]
    label_1 = raw.get('label_1', [{}])[0]
    label_2 = raw.get('label_2', [{}])[0]

    return {
        'cat_num': cat_num,
        'cat_name': cat_name,
        'topic': topic,
        'zip_file': zip_fname,
        'json_file': jf_name,
        'utterance': answer,
        'length': len(answer),
        'age': teller.get('나이', None),
        'gender': teller.get('성별', ''),
        'keyword': raw.get('keyword', ''),
        # label_1: 기억의 구체성 지표
        'l1_event': label_1.get('사건구체성', 0),
        'l1_time': label_1.get('시간적구체성', 0),
        'l1_space': label_1.get('공간적구체성', 0),
        'l1_subjective': label_1.get('주관적경험', 0),
        'l1_autobio': label_1.get('자서전적기억', 0),
        # label_2: 발화 특성
        'l2_emotion': label_2.get('감정', 0),
        'l2_person': label_2.get('인물', 0),
    }


SAMPLES_PER_ZIP = 2
rows = []
parse_errors = []

for zip_fname in all_zips:
    zip_path = os.path.join(LABEL_DIR, zip_fname)

    try:
        parts = zip_fname.split('.')
        cat_num = parts[0].split('_')[1]
        cat_name = parts[1].rsplit('_', 1)[0]
        topic_name = parts[-2] if parts[-1] == 'zip' else parts[-1]
    except Exception as e:
        parse_errors.append((zip_fname, str(e)))
        continue

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            json_files = [f for f in z.namelist() if f.endswith('.json')]
            for jf_name in json_files[:SAMPLES_PER_ZIP]:
                with z.open(jf_name) as jf:
                    raw = json.load(jf)
                row = extract_row(raw, cat_num, cat_name, topic_name, zip_fname, jf_name)
                if row:
                    rows.append(row)
    except Exception as e:
        parse_errors.append((zip_fname, str(e)))

df = pd.DataFrame(rows)
print(f'총 발화 수: {len(df)}개 (zip당 {SAMPLES_PER_ZIP}개 샘플)')
print(f'파싱 오류: {len(parse_errors)}개')
if parse_errors:
    for fname, err in parse_errors[:5]:
        print(f'  {fname}: {err}')

print('\n[대분류별 발화 수]')
print(df.groupby(['cat_num', 'cat_name']).size().reset_index(name='count'))

print('\n[발화 샘플 (대분류별 1개)]')
for cat_num in sorted(df['cat_num'].unique()):
    r = df[df['cat_num'] == cat_num].iloc[0]
    print(f'\n[{cat_num}] {r["cat_name"]} — {r["topic"]}')
    print(f'  {r["utterance"][:100]}')

## Cell 5. JSON 키 구조 재확인 (파싱 실패 시 수정)

Cell 4에서 utterance가 0개인 대분류가 있으면 해당 zip을 열어 실제 키 확인.

In [ ]:
# 대분류별 utterance 수 확인
empty_cats = df[df['utterance'].str.len() == 0]['cat_num'].unique() if len(df) > 0 else []

# 발화가 0개인 경우 raw 구조 재확인
if len(df) == 0:
    print('발화가 하나도 없음 — 구조 재확인')
    # 첫 번째 zip의 첫 번째 JSON 전체 출력
    sample_zip_path = os.path.join(LABEL_DIR, all_zips[0])
    with zipfile.ZipFile(sample_zip_path, 'r') as z:
        json_files = [f for f in z.namelist() if f.endswith('.json')]
        with z.open(json_files[0]) as jf:
            raw = json.load(jf)

    print('타입:', type(raw).__name__)
    if isinstance(raw, dict):
        print('키:', list(raw.keys()))
        for k, v in raw.items():
            print(f'  [{k}] ({type(v).__name__}): ', end='')
            if isinstance(v, list) and v:
                print(f'{len(v)}개, 첫 항목:', v[0] if not isinstance(v[0], dict) else list(v[0].keys()))
            elif isinstance(v, dict):
                print(list(v.keys())[:5])
            else:
                print(v)
    elif isinstance(raw, list):
        print(f'리스트 {len(raw)}개')
        if raw:
            print('첫 항목:', raw[0])
else:
    print('파싱 성공!')
    print('\n[대분류별 발화 샘플 (각 2개)]')
    for cat_num in sorted(df['cat_num'].unique()):
        subset = df[df['cat_num'] == cat_num]
        cat_name = subset['cat_name'].iloc[0]
        print(f'\n=== [{cat_num}] {cat_name} ===')
        for _, r in subset.head(2).iterrows():
            print(f'  [{r["topic"]}] {r["utterance"][:80]}')

## Cell 6. 전체 데이터 로드 (대분류별 주요 발화 수집)

Cell 4·5에서 구조 확인 완료 후, 각 zip의 JSON 5개씩 로드하여
카테고리 매핑 기준 도출에 충분한 샘플 확보.

In [ ]:
SAMPLES_PER_ZIP_FULL = 5  # zip당 JSON 샘플 수

rows_full = []

for zip_fname in all_zips:
    zip_path = os.path.join(LABEL_DIR, zip_fname)

    try:
        parts = zip_fname.split('.')
        cat_num = parts[0].split('_')[1]
        cat_name = parts[1].rsplit('_', 1)[0]
        topic_name = parts[-2] if parts[-1] == 'zip' else parts[-1]
    except Exception:
        continue

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            json_files = [f for f in z.namelist() if f.endswith('.json')]
            for jf_name in json_files[:SAMPLES_PER_ZIP_FULL]:
                with z.open(jf_name) as jf:
                    raw = json.load(jf)
                # extract_row 사용 — label_1 컬럼 포함
                row = extract_row(raw, cat_num, cat_name, topic_name, zip_fname, jf_name)
                if row and row['length'] >= 30:
                    rows_full.append(row)
    except Exception:
        continue

df_full = pd.DataFrame(rows_full)
print(f'전체 발화 (30자 이상): {len(df_full)}개')
print(f'컬럼 목록: {list(df_full.columns)}')
print('\n[대분류별 발화 수]')
print(df_full.groupby(['cat_num', 'cat_name'])['utterance'].count())
print('\n[발화 길이 통계]')
print(df_full['length'].describe())
print('\n[나이 분포]')
print(df_full['age'].describe())

## Cell 7. 카테고리 매핑 기준 도출

025번 대분류 5개 → extract-memory 카테고리 7개 매핑:

| 025 대분류 | 주제 예시 | 매핑 카테고리 |
|---|---|---|
| 01 | 미정 (Cell 2 확인 필요) | 감정경험 / 가치관 |
| 02 | 미정 | 장소추억 / 감정경험 |
| 03 | 사물 (책·꽃·음식 등) | 취미일상 |
| 04 | 장소 (산·바다·학교 등) | 장소추억 |
| 05 | 관계 및 사건 | 관계사건 |

→ 각 대분류·주제별 대표 발화 1~2개를 few-shot 예시로 선별

In [ ]:
# extract-memory 카테고리 7개 (TASK-03 기준)
CATEGORIES = {
    '감정경험': '기쁨·슬픔·그리움·후회 등 감정이 수반된 기억',
    '관계사건': '가족·친구·이웃과의 구체적 사건',
    '장소추억': '산·바다·학교·집 등 장소 기반 기억',
    '취미일상': '반복적인 취미·습관·일과',
    '건강': '신체·병원·약 관련',
    '가치관': '삶의 교훈·신념',
    '일정': '날짜가 있는 예정 사건',
}

# 025번 대분류 → 카테고리 기본 매핑 (Cell 2 출력 기반 확정)
# 01: 감정-긍정 및 중립 → 감정경험
# 02: 감정-부정         → 감정경험
# 03: 사물              → 취미일상 (선물·책·꽃·음식 등 일상 사물 관련 구술)
# 04: 장소              → 장소추억
# 05: 관계 및 사건      → 관계사건
CAT_NUM_TO_CATEGORY = {
    '01': '감정경험',
    '02': '감정경험',
    '03': '취미일상',
    '04': '장소추억',
    '05': '관계사건',
}

# 주제명으로 세분화 (건강·가치관·일정은 기본 매핑에 없으므로 주제 키워드로 보완)
TOPIC_KEYWORD_MAP = {
    '병원': '건강',
    '힘들다': '건강',
    # 장소 주제 중 병원은 건강으로
    # 관계 주제 중 일부는 가치관
    '그립다': '감정경험',
    '후회하다': '감정경험',
    '미안하다': '감정경험',
    '망설이다': '가치관',
}

def map_to_category(cat_num, topic, utterance):
    """대분류 번호·주제명 기반으로 extract-memory 카테고리 결정"""
    # 주제명 직접 매핑 우선
    for keyword, cat in TOPIC_KEYWORD_MAP.items():
        if keyword in topic:
            return cat
    # 병원 주제는 장소 대분류지만 건강으로 분류
    if topic == '병원':
        return '건강'
    # 대분류 번호 기본 매핑
    return CAT_NUM_TO_CATEGORY.get(cat_num, '감정경험')

# 매핑 적용
df_full['category'] = df_full.apply(
    lambda r: map_to_category(r['cat_num'], r['topic'], r['utterance']),
    axis=1
)

print('[카테고리별 발화 수]')
print(df_full['category'].value_counts())

print('\n[대분류·주제·카테고리 매핑 확인]')
mapping_check = df_full.groupby(['cat_num', 'cat_name', 'topic', 'category']).size().reset_index(name='count')
for _, r in mapping_check.iterrows():
    print(f'  [{r["cat_num"]}] {r["topic"]:12s} → {r["category"]} ({r["count"]}개)')

print('\n[카테고리별 발화 샘플 (각 1개)]')
for cat in CATEGORIES:
    subset = df_full[df_full['category'] == cat]
    if len(subset) == 0:
        print(f'\n[{cat}] 발화 없음')
        continue
    r = subset.iloc[0]
    print(f'\n[{cat}]')
    print(f'  주제: {r["topic"]} | {r["utterance"][:80]}')

## Cell 8. few-shot 예시 선별 및 JSON 저장

extract-memory few-shot 형식:
```json
[
  {
    "utterance": "...",
    "category": "관계사건",
    "text": "...(요약된 메모리 텍스트)",
    "emoji": "👨‍👩‍👧",
    "topic": "부모",
    "cat_num": "05",
    "note": "..."
  },
  ...
]
```

카테고리별 2개씩 선별 → 총 14개 (7카테고리 × 2)

In [ ]:
# few-shot 예시 선별 기준:
# 1. 길이 50~200자
# 2. label_1 quality 점수 높은 순
# 3. 개인정보 마스킹(***) 발화 제외
# 4. 감정 부재("별로 못 느껴", "없고" 등) 발화 제외
# 5. 주제와 카테고리 불일치 발화 제외 (topic 키워드가 발화에 없는 경우 하위 우선순위)
# 6. text 요약: 20자 이상 확보

CATEGORY_EMOJI = {
    '감정경험': '💭',
    '관계사건': '👨‍👩‍👧',
    '장소추억': '🏡',
    '취미일상': '🌿',
    '건강': '🏥',
}

# str.contains 경고 방지: regex=True, na=False 명시
ORAL_PATTERN = r'었지|거든요|더라고요|잖아요|이었어|했지요|그랬어|했는데|었는데'
MASK_PATTERN = r'\*{3,}'          # 개인정보 마스킹 발화 제외
NEGATION_PATTERN = r'별로.*못|없고|안 하고|별로.*없|모르겠'  # 감정 부재 표현

TARGET_CATS = ['감정경험', '관계사건', '장소추억', '취미일상', '건강']
SAMPLES_PER_CAT = 2

fewshot_025 = []

for cat in TARGET_CATS:
    subset = df_full[
        (df_full['category'] == cat) &
        (df_full['length'] >= 50) &
        (df_full['length'] <= 200) &
        # 마스킹 발화 제외
        ~df_full['utterance'].str.contains(MASK_PATTERN, regex=True, na=False) &
        # 부정/감정 부재 발화 제외
        ~df_full['utterance'].str.contains(NEGATION_PATTERN, regex=True, na=False)
    ].copy()

    if len(subset) == 0:
        print(f'[{cat}] 해당 발화 없음')
        continue

    subset['quality'] = (
        subset['l1_event'] + subset['l1_subjective'] + subset['l1_autobio']
    )
    subset['is_oral'] = subset['utterance'].str.contains(
        ORAL_PATTERN, regex=True, na=False
    )

    subset = subset.sort_values(
        ['quality', 'is_oral', 'length'],
        ascending=[False, False, True]
    )

    selected = subset.head(SAMPLES_PER_CAT)

    for _, r in selected.iterrows():
        # text 요약: 마침표 또는 쉼표 기준 첫 절, 최소 20자 확보
        raw_utt = r['utterance']
        # 마침표 기준 분리
        sentences = [s.strip() for s in raw_utt.replace('。', '.').split('.') if len(s.strip()) >= 10]
        if sentences:
            text_summary = sentences[0][:45] + ('...' if len(sentences[0]) > 45 else '')
        else:
            text_summary = raw_utt[:45] + '...'

        fewshot_025.append({
            'utterance': raw_utt,
            'category': cat,
            'text': text_summary,
            'emoji': CATEGORY_EMOJI[cat],
            'topic': r['topic'],
            'cat_num': r['cat_num'],
            'quality_score': int(r['quality']),
            'note': '025번 구술 데이터 — TASK-03 few-shot 초안 (수동 검토 필요)',
        })

print(f'few-shot 예시 총 {len(fewshot_025)}개\n')
for item in fewshot_025:
    print(f'[{item["category"]}] 주제:{item["topic"]} / quality:{item["quality_score"]}')
    print(f'  발화: {item["utterance"][:100]}')
    print(f'  text: "{item["text"]}" / emoji: {item["emoji"]}')
    print()

## Cell 9. Drive 저장

In [ ]:
import json

OUTPUT_DRIVE = '/content/drive/MyDrive/Dadam_dataSet/025_fewshot_category.json'
with open(OUTPUT_DRIVE, 'w', encoding='utf-8') as f:
    json.dump(fewshot_025, f, ensure_ascii=False, indent=2)

print(f'Drive 저장 완료: {OUTPUT_DRIVE}')
print(f'총 {len(fewshot_025)}개 few-shot 예시')
print('\n[저장 내용 미리보기]')
for item in fewshot_025:
    print(f'  [{item["category"]}] {item["utterance"][:60]}...')

## Cell 10. 로컬 다운로드

→ `prompt/fewshot/025_fewshot_category.json` 으로 저장

In [ ]:
from google.colab import files

LOCAL_PATH = '/content/025_fewshot_category.json'
with open(LOCAL_PATH, 'w', encoding='utf-8') as f:
    json.dump(fewshot_025, f, ensure_ascii=False, indent=2)

files.download(LOCAL_PATH)
print('다운로드 완료 → prompt/fewshot/025_fewshot_category.json 에 저장')